# P1-24 · Consolidate per-image results of all finished runs (CPU, about 2 min)
**Runtime: CPU.** No training. Reads `MyDrive/P1_v2/runs/*/` (read only) and writes four compact tables to `MyDrive/P1_v2/p1all_*.csv.gz`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, pandas as pd
OUT = '/content/drive/MyDrive/P1_v2'
T, E, M, R, H = [], [], [], [], []
for d in sorted(glob.glob(f'{OUT}/runs/*_seed*')):
    if not os.path.exists(d + '/run.json'): continue
    run = os.path.basename(d); exp, model, s = run.split('_'); seed = int(s[4:])
    r = json.load(open(d + '/run.json')); r.update(run=run, exp=exp, model=model, seed=seed); R.append(r)
    for name, L in (('test_per_image.csv', T), ('dmid_per_image.csv', E), ('mias_per_image.csv', M), ('history.csv', H)):
        f = f'{d}/{name}'
        if os.path.exists(f):
            t = pd.read_csv(f); t.insert(0, 'run', run); t.insert(1, 'exp', exp); t.insert(2, 'model', model); t.insert(3, 'seed', seed); L.append(t)
for name, L in (('test', T), ('dmid', E), ('mias', M), ('history', H)):
    df = pd.concat(L, ignore_index=True); df.to_csv(f'{OUT}/p1all_{name}.csv.gz', index=False, compression='gzip', float_format='%.17g')
    print(name, df.shape, '| runs:', df.run.nunique())
R = pd.DataFrame(R); R.to_csv(f'{OUT}/p1all_runs.csv.gz', index=False, compression='gzip')
print(R.groupby(['exp', 'model']).seed.apply(lambda s: sorted(s)).to_string())
print('DONE')